# Stage 3 -- Feature Naming: Combined Means & Combined Full Moments

## Purpose
Two short utility notebooks that build human-readable factor inventory CSVs for the two combined (daily + monthly) model-ready tables. These extend the logic of Notebooks 01 and 02 by handling features from all four panels simultaneously: Panel A (stock daily), Panel B (stock monthly), Panel C (macro daily), and Panel D (macro monthly). Monthly features are distinguished from daily features by a `monthly_` prefix applied during the Stage 3 merge.

---

## Notebook 03: Combined Means Factor Inventory
**Input:** `Data/Data_Collection/Final/Stage_3_Model_Ready/model_market_combined_means.parquet`
**Inventories:** `stock_daily_factor_inventory_final.csv`, `macro_daily_factor_inventory_final.csv`, `stock_monthly_factor_inventory_final.csv`, `macro_monthly_factor_inventory_final.csv`

### Logic
Column names are read from the parquet schema only (no data loaded). Both targets (`target_daily_return`, `target_monthly_return`) and `date` are excluded. The remaining features are classified by their naming convention:

- **Monthly features** (column name starts with `monthly_`): the `monthly_` prefix is stripped to recover the base name, which is looked up first in the stock monthly inventory, then in the macro monthly inventory.
- **Daily features** (no `monthly_` prefix): looked up first in the stock daily inventory, then in the macro daily inventory. The `stock_skew_chg_5d` rename conflict is handled as in Notebooks 01 and 02.

Unmatched columns are flagged. Breakdown by frequency (daily / monthly) and panel is printed.

### Output Columns
`column`, `base_factor`, `frequency` (daily / monthly), `panel` (A / B / C / D), `agg_method` (cwmean / raw level), `source`, `category`, `description`

**Output:** `Data/Data_Collection/Final/Stage_3_Model_Ready/combined_means_factor_inventory.csv`

---


In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

# Load feature names from combined means
BASE = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready')
schema = pq.read_schema(BASE / 'model_market_combined_means.parquet')
all_cols = [f.name for f in schema]
features = [c for c in all_cols if c not in ['date', 'target_daily_return', 'target_monthly_return']]

# Load all four inventories
INV_DIR = Path('../../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering')
stock_daily_inv = pd.read_csv(INV_DIR / 'stock_daily_factor_inventory_final.csv')
macro_daily_inv = pd.read_csv(INV_DIR / 'macro_daily_factor_inventory_final.csv')
stock_monthly_inv = pd.read_csv(INV_DIR / 'stock_monthly_factor_inventory_final.csv')
macro_monthly_inv = pd.read_csv(INV_DIR / 'macro_monthly_factor_inventory_final.csv')

stock_daily_map = stock_daily_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_daily_map = macro_daily_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
stock_monthly_map = stock_monthly_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')
macro_monthly_map = macro_monthly_inv.set_index('column')[['source', 'category', 'description']].to_dict('index')

# Handle renamed conflict
stock_daily_map['stock_skew_chg_5d'] = stock_daily_map.get('skew_chg_5d', {})

rows = []
matched = 0
unmatched = []

for f in features:
    # Monthly features have 'monthly_' prefix
    if f.startswith('monthly_'):
        base = f[len('monthly_'):]  # strip prefix
        
        if base in stock_monthly_map:
            info = stock_monthly_map[base]
            rows.append({
                'column': f,
                'base_factor': base,
                'frequency': 'monthly',
                'panel': 'B (stock monthly)',
                'agg_method': 'cwmean',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        elif base in macro_monthly_map:
            info = macro_monthly_map[base]
            rows.append({
                'column': f,
                'base_factor': base,
                'frequency': 'monthly',
                'panel': 'D (macro monthly)',
                'agg_method': 'raw level',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        else:
            rows.append({
                'column': f, 'base_factor': base, 'frequency': 'monthly',
                'panel': '???', 'agg_method': '', 'source': '', 'category': '', 'description': '',
            })
            unmatched.append(f)
    else:
        # Daily features — same logic as daily means inventory
        if f in stock_daily_map:
            info = stock_daily_map[f]
            rows.append({
                'column': f,
                'base_factor': f,
                'frequency': 'daily',
                'panel': 'A (stock daily)',
                'agg_method': 'cwmean',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        elif f in macro_daily_map:
            info = macro_daily_map[f]
            rows.append({
                'column': f,
                'base_factor': f,
                'frequency': 'daily',
                'panel': 'C (macro daily)',
                'agg_method': 'raw level',
                'source': info.get('source', ''),
                'category': info.get('category', ''),
                'description': info.get('description', ''),
            })
            matched += 1
        else:
            rows.append({
                'column': f, 'base_factor': f, 'frequency': 'daily',
                'panel': '???', 'agg_method': '', 'source': '', 'category': '', 'description': '',
            })
            unmatched.append(f)

result = pd.DataFrame(rows)

print(f"Total features: {len(features)}")
print(f"Matched: {matched}")
print(f"Unmatched: {len(unmatched)}")
if unmatched:
    print(f"\nUnmatched columns:")
    for c in unmatched:
        print(f"  {c}")

print(f"\nBy frequency:")
print(result['frequency'].value_counts().to_string())

print(f"\nBy panel:")
print(result['panel'].value_counts().to_string())

# Save
out_path = Path('../../../../Data/Data_Collection/Final/Stage_3_Model_Ready/combined_means_factor_inventory.csv')
result.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"  {len(result)} rows")

Total features: 722
Matched: 722
Unmatched: 0

By frequency:
frequency
daily      398
monthly    324

By panel:
panel
C (macro daily)      208
B (stock monthly)    191
A (stock daily)      190
D (macro monthly)    133

Saved: ..\..\..\..\Data\Data_Collection\Final\Stage_3_Model_Ready\combined_means_factor_inventory.csv
  722 rows
